# C08 Cosine classifier on A100

Frozen MERT95/330 verified feature caches; 64-channel residual Conv1d with three GELU layers; masked max pooling; cosine classifier with fixed scale 10. Label smoothing 0.05 applies only to training. AdamW initial lr 1e-4, weight decay 0.01; ReduceLROnPlateau halves lr after five stale validation epochs, floor 1e-6. Dropout 0.1, crop 0.7, feature mask probability 0.3. Six head fits: seeds 41, 42, 43 per backbone. Tune-only selection before historical test evaluation. Existing raw audio, assets and cache hashes are checked before execution. No new feature extraction is permitted by bootstrap. Two-hour compute ceiling; no official total-score claim.

In [ ]:
from temporal_backbone import main
main()

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
out=Path('results')
selection=json.loads((out/'selection.json').read_text())
lookup={r['name']:r for r in json.loads((out/'test_report.json').read_text())['results']}
summary=[]
for group in selection['summaries']:
    rows=[lookup[k] for k in group['candidates']]
    summary.append(dict(name=group['name'],tune_eer=group['music_eer_mean'],tune_bce=group['balanced_bce_mean'],
      test_music_eer=float(np.mean([r['music_eer'] for r in rows])),
      test_file_eer=float(np.mean([r['metrics']['metrics']['file_eer'] for r in rows]))))
fig,axes=plt.subplots(1,3,figsize=(12,4),layout='constrained')
for ax,key,title in zip(axes,['tune_eer','test_music_eer','test_file_eer'],['Tune music EER','Historical music EER','Historical file EER']):
    bars=ax.bar(['MERT95 / 768','MERT330 / 1024'],[r[key] for r in summary],color=['#6c8eb4','#ee9b49'])
    ax.bar_label(bars,fmt='%.3f');ax.set_ylim(0,.65);ax.set_title(title+' (lower better)');ax.grid(axis='y',alpha=.2)
fig.savefig(out/'comparison.png',dpi=170);plt.show()
(out/'summary.json').write_text(json.dumps(dict(groups=summary,winner=selection['winner']['name']),indent=2))
print(json.dumps(summary,indent=2))